In [8]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
import xgboost, lightgbm, shap, statsmodels, matplotlib, seaborn
print("all libraries ok")

all libraries ok


In [2]:
SEED = 42
df = pd.read_csv("bedtime_screentime_sleep_debt.csv")

print(df.shape)
print(df.dtypes)
print("missing:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print("duplicate user_id:", df["user_id"].duplicated().sum())

(8500, 18)
user_id                         str
age                           int64
gender                          str
occupation_type                 str
chronotype                      str
bedtime_phone_minutes         int64
primary_bedtime_app             str
screen_brightness_pct         int64
blue_light_filter_active      int64
caffeine_post_5pm_mg          int64
physical_activity_min         int64
sleep_latency_min           float64
total_sleep_hours           float64
deep_sleep_pct              float64
rem_sleep_pct               float64
morning_alarm_snoozes         int64
next_day_fatigue_score      float64
sleep_debt_category             str
dtype: object
missing: 0
duplicate rows: 0
duplicate user_id: 0


In [3]:
df = df.drop(columns=["user_id", "sleep_debt_category"])
print(df.shape)   # (8500, 16) before the flags; (8500, 18) after adding them

(8500, 16)


In [4]:
df["fatigue_at_floor"] = (df["next_day_fatigue_score"] <= 1.0).astype(int)
df["fatigue_at_cap"]   = (df["next_day_fatigue_score"] >= 10).astype(int)

print(df["next_day_fatigue_score"].min(), df["next_day_fatigue_score"].max())
print(df[["fatigue_at_floor", "fatigue_at_cap"]].sum())    # how many rows
print(df[["fatigue_at_floor", "fatigue_at_cap"]].mean())   # share of rows
print(df.shape)

1.0 10.0
fatigue_at_floor    2110
fatigue_at_cap       314
dtype: int64
fatigue_at_floor    0.248235
fatigue_at_cap      0.036941
dtype: float64
(8500, 18)


In [5]:
habits    = ["bedtime_phone_minutes", "caffeine_post_5pm_mg", "physical_activity_min",
             "screen_brightness_pct", "blue_light_filter_active"]
context   = ["age", "gender", "occupation_type", "chronotype"]
app       = ["primary_bedtime_app"]
mediators = ["sleep_latency_min", "total_sleep_hours"]
stage1_targets = mediators + ["morning_alarm_snoozes"]
outcome   = "next_day_fatigue_score"
flags     = ["fatigue_at_floor", "fatigue_at_cap"]
unused    = ["deep_sleep_pct", "rem_sleep_pct"]   # kept in df, not used as inputs

used = set(habits + context + app + stage1_targets + [outcome] + flags + unused)
print("unassigned columns:", set(df.columns) - used)   # should print set()

stage1_features = habits + context + app
stage2_features = habits + context + app + mediators

unassigned columns: set()


In [6]:
binary_cols = ["blue_light_filter_active"]
cat_cols    = ["gender", "occupation_type", "chronotype", "primary_bedtime_app"]

def build_preprocessor(features):
    num  = [c for c in features if c not in cat_cols + binary_cols]
    bin_ = [c for c in features if c in binary_cols]
    cat  = [c for c in features if c in cat_cols]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("bin", "passthrough", bin_),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
    ])

In [7]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["fatigue_at_floor"]
)
print(train_df.shape, test_df.shape)   # expect (6800, 18) and (1700, 18)
print(train_df["fatigue_at_floor"].mean(), test_df["fatigue_at_floor"].mean())  # both near 0.248

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Quick pipeline test: Stage 1, predict sleep_latency_min with Ridge
pipe = Pipeline([("prep", build_preprocessor(stage1_features)), ("model", Ridge(alpha=1.0))])
scores = cross_val_score(pipe, train_df[stage1_features], train_df["sleep_latency_min"],
                         cv=cv, scoring="r2")
print("CV R2:", scores.round(3), "mean:", scores.mean().round(3))

# Save outputs for the group
df.to_csv("clean_df.csv", index=False)
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
joblib.dump(build_preprocessor(stage1_features), "preprocessor_stage1.joblib")
joblib.dump(build_preprocessor(stage2_features), "preprocessor_stage2.joblib")
joblib.dump({"stage1_features": stage1_features, "stage2_features": stage2_features,
             "mediators": mediators, "stage1_targets": stage1_targets,
             "outcome": outcome, "seed": SEED}, "column_groups.joblib")
print("saved")

(6800, 18) (1700, 18)
0.24823529411764705 0.24823529411764705
CV R2: [0.84  0.833 0.837 0.835 0.846] mean: 0.838
saved
